In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [45]:
df = pd.read_csv('cleaned_data.csv')
df.sample()

,age,RevolvingUtilizationOfUnsecuredLines,NumberOfTime30-59DaysPastDueNotWorse,NumberOfTime60-89DaysPastDueNotWorse,NumberOfTimes90DaysLate,DebtRatio,DebtRatioMissing,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberRealEstateLoansOrLines,NumberOfDependents,SeriousDlqin2yrs
98944,97,0.001583,0,1,0,0.001999,0,1500.0,15,0,0.0,0


In [46]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [47]:
from sklearn.model_selection import train_test_split

In [48]:
X_train, X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [49]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [50]:
from sklearn.model_selection import cross_val_score

In [51]:
import optuna

In [12]:
def objective(trial):

    classifier_model = trial.suggest_categorical('classifier',['RFC','GB'])

    if classifier_model=='RFC':
        n_estimators = trial.suggest_int('estimators',20,150,step=2)
        criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
        max_depth = trial.suggest_int('max_depth',3,7)
        min_samples_split = trial.suggest_int('min_samples_split',2,10,step=2)
        min_samples_leaf = trial.suggest_int('min_samples_leaf',2,16,step=2)
        max_features = trial.suggest_categorical('max_features',['sqrt','log2'])
        bootstrap = trial.suggest_categorical('bootstrap',[True,False])

        model = RandomForestClassifier(n_estimators=n_estimators,criterion=criterion,max_depth=max_depth,min_samples_split=min_samples_split,
                                      min_samples_leaf=min_samples_leaf,max_features=max_features,bootstrap=bootstrap,random_state=42,
                                      class_weight={0:1,1:15.74})
        
        score = cross_val_score(model,X_train,y_train,cv=3,scoring='recall').mean()

    elif classifier_model == 'GB':

        n_estimators=trial.suggest_int('n_estimators',50,150,step=2)
        learning_rate=trial.suggest_float('learning_rate',0.01,0.3,log=True)
        min_samples_split = trial.suggest_int('min_samples_split',2,10,step=2)
        min_samples_leaf = trial.suggest_int('min_samples_leaf',2,16,step=2)

        model = GradientBoostingClassifier(n_estimators=n_estimators,learning_rate=learning_rate,
                                           min_samples_split=min_samples_split,min_samples_leaf=min_samples_leaf,random_state=42)

        sample_weight = np.where(y_train==1,15.74,1.0)
        
        score = cross_val_score(model, X_train, y_train, cv=3, scoring='recall',params={"sample_weight": sample_weight}).mean()
        
    return score  # Return the recall score for Optuna to maximize

In [13]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize recall
study.optimize(objective, n_trials=125)  # Run 125 trials to find the best hyperparameters

[I 2026-08-17 21:04:34,329] A new study created in memory with name: no-name-d302d841-9454-43a9-a556-f3e7036c3a70
[I 2026-08-17 21:04:39,536] Trial 0 finished with value: 0.7754160487724501 and parameters: {'classifier': 'RFC', 'estimators': 64, 'criterion': 'log_loss', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 12, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: 0.7754160487724501.
[I 2026-08-17 21:04:43,824] Trial 1 finished with value: 0.7457571263799637 and parameters: {'classifier': 'RFC', 'estimators': 24, 'criterion': 'log_loss', 'max_depth': 5, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'bootstrap': False}. Best is trial 0 with value: 0.7754160487724501.
[I 2026-08-17 21:04:52,672] Trial 2 finished with value: 0.7805239742956006 and parameters: {'classifier': 'RFC', 'estimators': 110, 'criterion': 'entropy', 'max_depth': 3, 'min_samples_split': 6, 'min_samples_leaf': 16, 'max_features': 'log2', 'bootstrap': T

In [14]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.8095238095238094
Best hyperparameters: {'classifier': 'GB', 'n_estimators': 60, 'learning_rate': 0.010165112224109157, 'min_samples_split': 6, 'min_samples_leaf': 14}


In [52]:
GB_model = GradientBoostingClassifier(n_estimators=60, learning_rate=0.010165112224109157, min_samples_split=6, min_samples_leaf=14)

In [53]:
sample_weight = np.where(y_train==1,15.74,1.0)

In [54]:
GB_model.fit(X_train,y_train,sample_weight=sample_weight)

GradientBoostingClassifier(learning_rate=0.010165112224109157,
                           min_samples_leaf=14, min_samples_split=6,
                           n_estimators=60)

In [55]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [56]:
y_pred = GB_model.predict(X_test)

In [57]:
accuracy_score(y_test,y_pred)

0.6826418693371483

In [58]:
confusion_matrix(y_test,y_pred)

array([[21311, 10293],
       [  355,  1593]], dtype=int64)

In [60]:
print('Classification report')
print(classification_report(y_test,y_pred))

Classification report
              precision    recall  f1-score   support

           0       0.98      0.67      0.80     31604
           1       0.13      0.82      0.23      1948

    accuracy                           0.68     33552
   macro avg       0.56      0.75      0.52     33552
weighted avg       0.93      0.68      0.77     33552



In [61]:
import pickle

# save using pickle

with open("GB_model.pkl", "wb") as f:
    pickle.dump(GB_model, f)